# PFAS Method Validation Checklist & Gap Analysis

**Method**: SOP E-AED-PEB-SOP-1908-1 (Waters Xevo TQD) → Shimadzu LCMS-8060  
**Instrument**: Shimadzu Nexera X2 / LCMS-8060  

This notebook loads the validation checklist and gap analysis YAML files,
displays interactive progress tracking, and allows updating status.

In [ ]:
from pathlib import Path
import yaml
import pandas as pd
from IPython.display import display, HTML

DATA_DIR = Path("../data/methods")
CHECKLIST_PATH = DATA_DIR / "validation_checklist.yaml"
GAP_PATH = DATA_DIR / "gap_analysis.yaml"

def load_checklist():
    with open(CHECKLIST_PATH) as f:
        return yaml.safe_load(f)

def load_gap_analysis():
    with open(GAP_PATH) as f:
        return yaml.safe_load(f)

def save_checklist(data):
    with open(CHECKLIST_PATH, "w") as f:
        yaml.dump(data, f, default_flow_style=False, sort_keys=False, width=120)

def save_gap_analysis(data):
    with open(GAP_PATH, "w") as f:
        yaml.dump(data, f, default_flow_style=False, sort_keys=False, width=120)

checklist = load_checklist()
gap = load_gap_analysis()
print(f"Loaded {len(checklist['phases'])} validation phases")
print(f"Loaded gap analysis with {len(gap['lc_parameters'])} LC params, "
      f"{len(gap['ms_source_parameters'])} MS params, "
      f"{len(gap['mrm_transitions'])} MRM transitions")

## Overall Progress

In [ ]:
STATUS_COLORS = {
    "pass": "#28a745",
    "fail": "#dc3545",
    "pending": "#6c757d",
    "na": "#17a2b8",
}

def progress_bar_html(done, total, label="", width=400):
    pct = (done / total * 100) if total > 0 else 0
    color = "#28a745" if pct == 100 else "#ffc107" if pct > 0 else "#6c757d"
    return (
        f'<div style="margin: 4px 0;">'
        f'<span style="display:inline-block;width:240px;">{label}</span>'
        f'<div style="display:inline-block;width:{width}px;background:#e9ecef;'
        f'border-radius:4px;overflow:hidden;vertical-align:middle;">'
        f'<div style="width:{pct:.0f}%;background:{color};height:20px;"></div>'
        f'</div>'
        f' <span style="margin-left:8px;">{done}/{total} ({pct:.0f}%)</span>'
        f'</div>'
    )

def count_statuses(steps):
    counts = {"pass": 0, "fail": 0, "pending": 0, "na": 0}
    for s in steps:
        status = s.get("status", "pending")
        if status in counts:
            counts[status] += 1
    return counts

# Overall summary
all_steps = []
phase_summaries = []
for phase in checklist["phases"]:
    steps = phase["steps"]
    all_steps.extend(steps)
    counts = count_statuses(steps)
    phase_summaries.append({
        "phase": phase["phase"],
        "name": phase["name"],
        "total": len(steps),
        "pass": counts["pass"],
        "fail": counts["fail"],
        "pending": counts["pending"],
        "na": counts["na"],
        "done": counts["pass"] + counts["na"],
    })

total_steps = len(all_steps)
total_counts = count_statuses(all_steps)
total_done = total_counts["pass"] + total_counts["na"]

html = "<h3>Validation Progress</h3>"
html += progress_bar_html(total_done, total_steps, "<b>Overall</b>", width=500)
html += "<br>"
for ps in phase_summaries:
    label = f"Phase {ps['phase']}: {ps['name']}"
    html += progress_bar_html(ps["done"], ps["total"], label, width=500)

html += "<br><table style='border-collapse:collapse;margin-top:12px;'>"
html += "<tr><th style='padding:4px 12px;'>Status</th><th style='padding:4px 12px;'>Count</th></tr>"
for status, count in total_counts.items():
    color = STATUS_COLORS[status]
    html += (f"<tr><td style='padding:4px 12px;'>"
             f"<span style='display:inline-block;width:12px;height:12px;"
             f"background:{color};border-radius:2px;margin-right:6px;'></span>"
             f"{status.upper()}</td>"
             f"<td style='padding:4px 12px;text-align:center;'>{count}</td></tr>")
html += "</table>"

display(HTML(html))

## Validation Checklist by Phase

In [ ]:
def status_badge(status):
    color = STATUS_COLORS.get(status, "#6c757d")
    return (f'<span style="background:{color};color:white;padding:2px 8px;'
            f'border-radius:3px;font-size:12px;">{status.upper()}</span>')

for phase in checklist["phases"]:
    html = f"<h3>Phase {phase['phase']}: {phase['name']}</h3>"
    html += f"<p style='color:#666;margin-top:-8px;'>{phase['description'].strip()}</p>"
    html += ("<table style='border-collapse:collapse;width:100%;'>"
             "<tr style='background:#f8f9fa;'>"
             "<th style='padding:6px;border:1px solid #dee2e6;width:50px;'>ID</th>"
             "<th style='padding:6px;border:1px solid #dee2e6;'>Description</th>"
             "<th style='padding:6px;border:1px solid #dee2e6;width:200px;'>Acceptance Criteria</th>"
             "<th style='padding:6px;border:1px solid #dee2e6;width:80px;'>Reference</th>"
             "<th style='padding:6px;border:1px solid #dee2e6;width:80px;'>Status</th>"
             "<th style='padding:6px;border:1px solid #dee2e6;width:100px;'>Date</th>"
             "<th style='padding:6px;border:1px solid #dee2e6;width:150px;'>Notes</th>"
             "</tr>")
    for step in phase["steps"]:
        date_str = step.get("date") or ""
        notes = step.get("notes") or ""
        criteria = step.get("acceptance_criteria", "").strip()
        ref = step.get("reference", "")
        html += (f"<tr>"
                 f"<td style='padding:6px;border:1px solid #dee2e6;text-align:center;'>{step['id']}</td>"
                 f"<td style='padding:6px;border:1px solid #dee2e6;'>{step['description'].strip()}</td>"
                 f"<td style='padding:6px;border:1px solid #dee2e6;font-size:12px;'>{criteria}</td>"
                 f"<td style='padding:6px;border:1px solid #dee2e6;font-size:12px;'>{ref}</td>"
                 f"<td style='padding:6px;border:1px solid #dee2e6;text-align:center;'>{status_badge(step['status'])}</td>"
                 f"<td style='padding:6px;border:1px solid #dee2e6;text-align:center;'>{date_str}</td>"
                 f"<td style='padding:6px;border:1px solid #dee2e6;font-size:12px;'>{notes}</td>"
                 f"</tr>")
    html += "</table>"
    display(HTML(html))

## Gap Analysis Summary

In [ ]:
GAP_COLORS = {
    "verified": "#28a745",
    "transferred": "#28a745",
    "needs_optimization": "#ffc107",
    "not_applicable": "#17a2b8",
}

def gap_badge(status):
    color = GAP_COLORS.get(status, "#dc3545")
    text_color = "#000" if status == "needs_optimization" else "#fff"
    return (f'<span style="background:{color};color:{text_color};padding:2px 8px;'
            f'border-radius:3px;font-size:12px;">{status.replace("_", " ").upper()}</span>')

def render_gap_table(title, entries, columns):
    html = f"<h4>{title}</h4>"
    header_html = "".join(
        f"<th style='padding:6px;border:1px solid #dee2e6;'>{col}</th>" for col in columns
    )
    html += (f"<table style='border-collapse:collapse;width:100%;'>"
             f"<tr style='background:#f8f9fa;'>{header_html}</tr>")
    for entry in entries:
        cells = []
        for col in columns:
            key = col.lower().replace(" ", "_")
            val = entry.get(key, "")
            if key == "status":
                val = gap_badge(val)
            elif val is None:
                val = "—"
            cells.append(f"<td style='padding:6px;border:1px solid #dee2e6;font-size:13px;'>{val}</td>")
        html += f"<tr>{''.join(cells)}</tr>"
    html += "</table>"
    return html

# Gap analysis summary counts
all_gap_items = gap["lc_parameters"] + gap["ms_source_parameters"] + gap["mrm_transitions"]
gap_counts = {}
for item in all_gap_items:
    s = item.get("status", "unknown")
    gap_counts[s] = gap_counts.get(s, 0) + 1

summary_html = "<h3>Gap Analysis Status Summary</h3><table style='border-collapse:collapse;'>"
summary_html += "<tr><th style='padding:4px 12px;'>Status</th><th style='padding:4px 12px;'>Count</th></tr>"
for status, count in sorted(gap_counts.items()):
    summary_html += f"<tr><td style='padding:4px 12px;'>{gap_badge(status)}</td><td style='padding:4px 12px;text-align:center;'>{count}</td></tr>"
summary_html += f"<tr style='font-weight:bold;'><td style='padding:4px 12px;'>Total</td><td style='padding:4px 12px;text-align:center;'>{len(all_gap_items)}</td></tr>"
summary_html += "</table>"
display(HTML(summary_html))

In [ ]:
# LC Parameters
html = render_gap_table(
    "LC Parameters",
    gap["lc_parameters"],
    ["Parameter", "SOP_value", "Shimadzu_value", "Status", "Notes"]
)
display(HTML(html))

In [ ]:
# MS Source Parameters
html = render_gap_table(
    "MS Source Parameters",
    gap["ms_source_parameters"],
    ["Parameter", "SOP_value", "Shimadzu_value", "Status", "Notes"]
)
display(HTML(html))

In [ ]:
# MRM Transitions
mrm_df = pd.DataFrame(gap["mrm_transitions"])
mrm_display = mrm_df[[
    "compound", "compound_class", "sop_quantifier", "sop_ce",
    "shimadzu_ce", "sop_rt_min", "shimadzu_rt_min",
    "sensitivity_comparison", "status"
]].copy()
mrm_display.columns = [
    "Compound", "Class", "Quantifier", "SOP CE",
    "8060 CE", "SOP RT", "8060 RT",
    "Sensitivity", "Status"
]

def color_status(val):
    colors = {
        "verified": "background-color: #d4edda",
        "transferred": "background-color: #d4edda",
        "needs_optimization": "background-color: #fff3cd",
        "not_applicable": "background-color: #d1ecf1",
    }
    return colors.get(val, "background-color: #f8d7da")

styled = mrm_display.style.applymap(color_status, subset=["Status"])
display(styled)

In [ ]:
# Compounds not in SOP
non_sop = gap["compounds_not_in_sop"]
html = f"<h4>Compounds Not in SOP (Require Standalone Validation)</h4>"
html += f"<p style='color:#856404;background:#fff3cd;padding:8px;border-radius:4px;'>{non_sop['description'].strip()}</p>"
html += ("<table style='border-collapse:collapse;width:100%;'>"
         "<tr style='background:#f8f9fa;'>"
         "<th style='padding:6px;border:1px solid #dee2e6;'>Compound</th>"
         "<th style='padding:6px;border:1px solid #dee2e6;'>Class</th>"
         "<th style='padding:6px;border:1px solid #dee2e6;'>Quantifier</th>"
         "<th style='padding:6px;border:1px solid #dee2e6;'>Est. CE</th>"
         "<th style='padding:6px;border:1px solid #dee2e6;'>Est. RT</th>"
         "<th style='padding:6px;border:1px solid #dee2e6;'>IS</th>"
         "<th style='padding:6px;border:1px solid #dee2e6;'>Status</th>"
         "<th style='padding:6px;border:1px solid #dee2e6;'>Validation Required</th>"
         "</tr>")
for c in non_sop["compounds"]:
    html += (f"<tr>"
             f"<td style='padding:6px;border:1px solid #dee2e6;font-weight:bold;'>{c['compound']}</td>"
             f"<td style='padding:6px;border:1px solid #dee2e6;'>{c['compound_class']}</td>"
             f"<td style='padding:6px;border:1px solid #dee2e6;'>{c['quantifier']}</td>"
             f"<td style='padding:6px;border:1px solid #dee2e6;text-align:center;'>{c['estimated_ce']}</td>"
             f"<td style='padding:6px;border:1px solid #dee2e6;text-align:center;'>{c['estimated_rt_min']}</td>"
             f"<td style='padding:6px;border:1px solid #dee2e6;'>{c['internal_standard']}</td>"
             f"<td style='padding:6px;border:1px solid #dee2e6;text-align:center;'>{gap_badge(c['status'])}</td>"
             f"<td style='padding:6px;border:1px solid #dee2e6;font-size:12px;'>{c['validation_required'].strip()}</td>"
             f"</tr>")
html += "</table>"
display(HTML(html))

In [ ]:
# QC Criteria comparison
html = render_gap_table(
    "QC Acceptance Criteria",
    gap["qc_criteria"]["criteria"],
    ["Parameter", "SOP_value", "Shimadzu_value", "Status", "Notes"]
)
display(HTML(html))

## Update Checklist Status

Use the cell below to update the status of a checklist step.
Modify the `step_id`, `new_status`, and `note` variables, then run the cell.

In [ ]:
# === EDIT THESE VALUES ===
step_id = "1.1"           # Step ID to update (e.g., "1.1", "2.3", "5.2")
new_status = "pending"     # New status: "pending", "pass", "fail", or "na"
note = ""                  # Optional note to append
date = ""                  # Date string (e.g., "2025-01-15") or leave empty
# =========================

updated = False
for phase in checklist["phases"]:
    for step in phase["steps"]:
        if step["id"] == step_id:
            step["status"] = new_status
            if note:
                existing = step.get("notes", "")
                step["notes"] = f"{existing}; {note}".lstrip("; ") if existing else note
            if date:
                step["date"] = date
            updated = True
            print(f"Updated step {step_id}: status={new_status}")
            if note:
                print(f"  Note: {step['notes']}")
            if date:
                print(f"  Date: {date}")
            break
    if updated:
        break

if not updated:
    print(f"Step {step_id} not found!")
else:
    save_checklist(checklist)
    print(f"Saved to {CHECKLIST_PATH}")

## Update Gap Analysis Status

Use the cell below to update the status of a gap analysis entry.
Modify the variables and run.

In [ ]:
# === EDIT THESE VALUES ===
category = "mrm_transitions"  # "lc_parameters", "ms_source_parameters", "mrm_transitions", or "qc_criteria"
search_key = "compound"       # Key to search by ("parameter" for LC/MS/QC, "compound" for MRM)
search_value = "PFOA"         # Value to match
new_status = "verified"       # New status: "transferred", "needs_optimization", "verified"
shimadzu_value = None         # New shimadzu value (set to None to keep existing)
note = ""                     # Optional note to append
# =========================

entries = gap.get(category, [])
if category == "qc_criteria":
    entries = gap["qc_criteria"]["criteria"]

updated = False
for entry in entries:
    if entry.get(search_key) == search_value:
        entry["status"] = new_status
        if shimadzu_value is not None:
            if category == "mrm_transitions":
                entry["shimadzu_ce"] = shimadzu_value
            else:
                entry["shimadzu_value"] = shimadzu_value
        if note:
            existing = entry.get("notes", "")
            entry["notes"] = f"{existing}; {note}".lstrip("; ") if existing else note
        updated = True
        print(f"Updated {search_value} in {category}: status={new_status}")
        break

if not updated:
    print(f"{search_value} not found in {category}!")
else:
    save_gap_analysis(gap)
    print(f"Saved to {GAP_PATH}")

## Summary Statistics

In [ ]:
# Reload to get fresh data
checklist = load_checklist()
gap = load_gap_analysis()

# Checklist stats
all_steps = [s for p in checklist["phases"] for s in p["steps"]]
total = len(all_steps)
counts = count_statuses(all_steps)
complete = counts["pass"] + counts["na"]

print("=" * 50)
print("VALIDATION CHECKLIST SUMMARY")
print("=" * 50)
print(f"  Total steps:    {total}")
print(f"  Passed:         {counts['pass']}")
print(f"  Failed:         {counts['fail']}")
print(f"  Pending:        {counts['pending']}")
print(f"  N/A:            {counts['na']}")
print(f"  % Complete:     {complete/total*100:.0f}%")
print()

# Gap analysis stats
all_gaps = (gap["lc_parameters"] + gap["ms_source_parameters"]
            + gap["mrm_transitions"] + gap["qc_criteria"]["criteria"])
gap_total = len(all_gaps)
gap_counts = {}
for item in all_gaps:
    s = item.get("status", "unknown")
    gap_counts[s] = gap_counts.get(s, 0) + 1

verified = gap_counts.get("verified", 0) + gap_counts.get("transferred", 0)

print("=" * 50)
print("GAP ANALYSIS SUMMARY")
print("=" * 50)
print(f"  Total parameters:      {gap_total}")
for status, count in sorted(gap_counts.items()):
    print(f"  {status:24s} {count}")
print(f"  % Verified/Transferred: {verified/gap_total*100:.0f}%")
print(f"  Items needing work:    {gap_counts.get('needs_optimization', 0)}")